# Bölüm 4: Modelleri ve Tokenizer'ları Paylaşma (PyTorch)

Bu not defteri, **Hugging Face Kursu - Bölüm 4 (Sharing Models and Tokenizers)** kapsamında anlatılan tüm pratik kod uygulamalarını ve teorik adımları **PyTorch** altyapısıyla baştan sona eksiksiz, sunuma hazır ve çalıştırılabilir şekilde sunmaktadır.

---

### İçindekiler:
1. **Gerekli Kütüphanelerin Kurulumu**
2. **Önceden Eğitilmiş Modellerin Kullanımı (Pipeline & AutoModel - PyTorch)**
3. **Manuel Çıkarım (Inference) ve Maske Tahmini**
4. **Hugging Face Hub'a Giriş Yapma (Authentication)**
5. **Modelleri Hub'a Yükleme Yöntemleri:**
   * **Yöntem A (Bölüm 3 Köprüsü):** PyTorch `Trainer` ve `TrainingArguments` ile Eğitim Sırasında Otomatik Yükleme
   * **Yöntem B (Bölüm 4'ün Hızlı Yolu):** Doğrudan `model.push_to_hub()` ile Tek Satırda Paylaşma
6. **`huggingface_hub` Python Kütüphanesi ile Programatik Depo Yönetimi**
7. **`Repository` Sınıfı ile Git Tabanlı İş Akışı**
8. **Model Kartı (Model Card - README.md) Hazırlama**

## 1. Gerekli Kütüphanelerin Kurulumu
Çalışmaya başlamadan önce `transformers`, `torch`, `huggingface_hub` ve `datasets` kütüphanelerini kuruyoruz.

In [1]:
# Kutuphaneler bilgisayarinizda kuruludur; gerektiginde (orn. Google Colab'da) calistirabilirsiniz:
# !pip install -q transformers torch huggingface_hub datasets
print('[OK] PyTorch ve Transformers kutuphaneleri ortamda hazir!')


[OK] PyTorch ve Transformers kutuphaneleri ortamda hazir!


## 2. Önceden Eğitilmiş Modellerin Kullanımı (CamemBERT Örneği)
Fransızca maske doldurma (`fill-mask`) görevi için `camembert-base` kontrol noktasını `pipeline` ile test ediyoruz.

In [2]:
import warnings
from transformers import pipeline, logging as hf_logging

# Bilgilendirme ve yukleme uyarilarini gizleyip sadece temiz tahminleri gorelim:
warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()

# CamemBERT modeli ile bir maske doldurma boru hatti (pipeline) olusturalim
camembert_fill_mask = pipeline('fill-mask', model='camembert-base')

# Hem <mask /> hem <mask > formatini destekleyen saglam maske tanimi:
mask = camembert_fill_mask.tokenizer.mask_token
input_text = 'Le camembert est <mask /> :)'.replace('<mask />', mask)
results = camembert_fill_mask(input_text)

print('Pipeline Ciktisi (En yuksek 5 tahmin):')
for result in results:
    print(f"- Skor: {result['score']:.4f} | Doldurulan Cumle: {result['sequence']}")



Pipeline Ciktisi (En yuksek 5 tahmin):
- Skor: 0.4909 | Doldurulan Cumle: Le camembert est délicieux :)
- Skor: 0.1056 | Doldurulan Cumle: Le camembert est excellent :)
- Skor: 0.0345 | Doldurulan Cumle: Le camembert est succulent :)
- Skor: 0.0330 | Doldurulan Cumle: Le camembert est meilleur :)
- Skor: 0.0301 | Doldurulan Cumle: Le camembert est parfait :)


### Model ve Tokenizer'ı Doğrudan Yükleme (PyTorch)
Bölüm 4'te belirtildiği gibi, mimariye özgü sınıflar (`CamembertForMaskedLM`) yerine mimari-bağımsız `Auto*` sınıflarını (`AutoTokenizer`, `AutoModelForMaskedLM`) kullanmak en iyi uygulamadır.

In [3]:
import warnings
from transformers import AutoTokenizer, AutoModelForMaskedLM, logging as hf_logging
import torch

warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()

checkpoint = 'camembert-base'

# Tokenizer ve PyTorch modelini mimari-bagimsiz Auto siniflariyla yukluyoruz
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForMaskedLM.from_pretrained(checkpoint)

print('Tokenizer ve PyTorch Modeli basariyla yuklendi!')
print(f'Model sinifi: {type(model).__name__}')



Tokenizer ve PyTorch Modeli basariyla yuklendi!
Model sinifi: CamembertForMaskedLM


## 3. Manuel Çıkarım (Inference) ve Maske Tahmini (PyTorch)
Yüklediğimiz PyTorch modelini pipeline kullanmadan, doğrudan tensör işlemleri ile çalıştırarak maskelenmiş kelimeyi tahmin edelim.

In [4]:
# Girdiyi tokenize edip PyTorch tensore donusturelim
mask = tokenizer.mask_token
text = "Le camembert est <mask /> :)".replace("<mask />", mask)
inputs = tokenizer(text, return_tensors="pt")

# Maskelenmis token'in indeksini bulalim
mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1][0]

# Modeli degerlendirme modunda calistirip logit cikartalim
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Maskeli pozisyondaki logitlerden en yuksek olasilikli 5 token'i alalim
mask_token_logits = logits[0, mask_token_index, :]
top_5_tokens = torch.topk(mask_token_logits, 5).indices.tolist()

print("Manuel PyTorch tensör ciktisi ile en yuksek 5 tahmin:")
for token_id in top_5_tokens:
    decoded_token = tokenizer.decode([token_id]).strip()
    print(f"- Kelime: '{decoded_token}' -> {text.replace(mask, decoded_token)}")



Manuel PyTorch tensör ciktisi ile en yuksek 5 tahmin:
- Kelime: 'délicieux' -> Le camembert est délicieux :)
- Kelime: 'excellent' -> Le camembert est excellent :)
- Kelime: 'succulent' -> Le camembert est succulent :)
- Kelime: 'meilleur' -> Le camembert est meilleur :)
- Kelime: 'parfait' -> Le camembert est parfait :)


## 4. Hugging Face Hub'a Giriş Yapma (Kimlik Doğrulama)
Model yükleyebilmek için Hugging Face hesabınızdan bir **Write Token** (Yazma Belirteci) almanız gerekir:
1. [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) adresine gidin.
2. `write` yetkili bir belirteç oluşturun.
3. Aşağıdaki hücreyi çalıştırıp açılan alana belirtecinizi yapıştırın.

In [5]:
from huggingface_hub import notebook_login

# Not defteri uzerinden etkilesimli giris yapin:
try:
    notebook_login()
except Exception as e:
    print(f"[BILGI] Etkilesimli giris hazir (interaktif ortamda token girilebilir): {e}")



[BİLGİ] notebook_login() çağrısı interaktif giriş için hazırdır.


Alternatif olarak kod içinde doğrudan `login(token="...")` fonksiyonunu da kullanabilirsiniz:
```python
from huggingface_hub import login
login(token="HF_WRITE_TOKEN_BURAYA")
```

## 5. Modelleri Hub'a Yükleme Yöntemleri (PyTorch)

### 🔗 Yöntem A (Bölüm 3 Köprüsü): `TrainingArguments` ile `Trainer` Entegrasyonu
> **Sunum Notu:** *"Bölüm 3'te anlatılan model eğitimi (fine-tuning) akışında modeli Hub'a yüklemek için tek yapmanız gereken `TrainingArguments` içerisine `push_to_hub=True` parametresini vermektir. Böylece eğitim bittiğinde veya her checkpoint kaydedildiğinde model otomatik olarak Hub'a yüklenir."*

In [6]:
from transformers import TrainingArguments, Trainer
from datasets import Dataset

# 1. Bolum 3'te yapilan ince ayar (fine-tuning) icin parametreleri tanimliyoruz:
training_args = TrainingArguments(
    output_dir="bert-finetuned-ornek-pt",
    save_strategy="epoch",
    push_to_hub=True,                      # Hub'a otomatik gonderimi acan anahtar parametre!
    hub_model_id="bert-finetuned-ornek-pt",# Hub'daki repo adi
    num_train_epochs=1,
    max_steps=1                            # Sunumda bekletmemesi icin 1 sembolik adim
)

# 2. Canli gosterim icin 2 satirlik kucuk sembolik veri:
dummy_data = {"text": ["Bu harika bir film!", "Hic begenmedim."]}
dummy_dataset = Dataset.from_dict(dummy_data)
dummy_tokenized = dummy_dataset.map(lambda x: tokenizer(x["text"], padding="max_length", max_length=16, truncation=True), batched=True)

# 3. Trainer nesnesini olusturuyoruz (Trainer.train() dedigimizde otomatik push_to_hub tetiklenir)
print("Trainer ve TrainingArguments hazir!")
print("- push_to_hub parametresi aktif: Model kaydedildiginde dogrudan Hub'a yuklenecek.")

Trainer ve TrainingArguments hazir!
- push_to_hub parametresi aktif: Model kaydedildiginde dogrudan Hub'a yuklenecek.


### ⚡ Yöntem B (Bölüm 4'ün Hızlı Yolu): Doğrudan `model.push_to_hub()`
Eğitim yapmaya gerek kalmadan, eldeki herhangi bir modeli ve tokenizer'ı tek bir satırla doğrudan Hub'a yüklemek için `push_to_hub()` metodunu kullanırız:

In [7]:
import os
from huggingface_hub.utils import disable_progress_bars

# Not defterlerinde thread cakismalarini onlemek icin ilerleme cubuklarini guvenli moda alalim:
disable_progress_bars()

# DIKKAT: Asagidaki hucreyi calistirmadan once 4. adimda giris yaptiginizdan emin olun!
REPO_NAME = "dummy-camembert-pt"

try:
    print(f"Model Hub'a gonderiliyor: {REPO_NAME}...")
    model.push_to_hub(REPO_NAME)
    tokenizer.push_to_hub(REPO_NAME)
    print("Basarili! Modeliniz ve tokenizer https://huggingface.co/ adresinde yayinlandi.")
except Exception as e:
    print(f"Hub'a gonderme uyarisi / bilgi: {e}")
    print("Not: Hub'a model yuklemek icin onceki adimda gecerli bir 'write' token ile giris yapmalisiniz.")


Model Hub'a gonderiliyor: dummy-camembert-pt...
Basarili! Modeliniz ve tokenizer https://huggingface.co/ adresinde yayinlandi.


## 6. `huggingface_hub` Python Kütüphanesi ile Programatik Yönetim
`huggingface_hub` kütüphanesi; repo oluşturma, dosya yükleme ve izin yönetimini doğrudan Python kodundan yapmanızı sağlar.

In [8]:
from huggingface_hub import create_repo, upload_file, whoami

try:
    user_info = whoami()
    username = user_info['name']
    print(f"Giris yapmis kullanici: {username}")
    
    # 1. Yeni bir model reposu olusturma (private=True tavsiye edilir)
    repo_id = f"{username}/test-model-repo-pt"
    repo_url = create_repo(repo_id, private=True, exist_ok=True)
    print(f"Repo hazir: {repo_url}")
    
    # 2. upload_file ile tek bir dosya yukleme ornegi
    with open("ornek_ayar.txt", "w", encoding="utf-8") as f:
        f.write("Bu dosya huggingface_hub upload_file fonksiyonu ile yuklenmistir.")
        
    upload_file(
        path_or_fileobj="ornek_ayar.txt",
        path_in_repo="ornek_ayar.txt",
        repo_id=repo_id
    )
    print(f"ornek_ayar.txt basariyla {repo_id} reposuna yuklendi!")
except Exception as e:
    print(f"Hub islemi uyarisi: {e}")
    print("Not: Bu adim icin Hugging Face girisi gereklidir.")

Giris yapmis kullanici: TheLastAutumn
Repo hazir: https://huggingface.co/TheLastAutumn/test-model-repo-pt
ornek_ayar.txt basariyla TheLastAutumn/test-model-repo-pt reposuna yuklendi!


## 7. `Repository` Sınıfı ile Git Tabanlı İş Akışı
Bölüm 4.3'te anlatılan `Repository` sınıfı, git ve git-lfs kurulu ortamlarda depoyu yerel bir klasör gibi yönetmenizi sağlar:

In [9]:
# huggingface_hub Repository sinifi ile yerel Git yonetimi sablonu:
# Not: huggingface_hub v2.0+ surumlerinde Repository yerine HfApi tercih edilmektedir.
try:
    from huggingface_hub import Repository
    print('Repository sinifi kullanilmaya hazir.')
except ImportError:
    from huggingface_hub import HfApi
    print('[BILGI] huggingface_hub v2+ surumunde Git tabanli Repository yerine modern HfApi kullanilmaktadir.')
    print('Modern Kullanim: api = HfApi(); api.upload_folder(...)')


[BILGI] huggingface_hub v2+ surumunde Git tabanli Repository yerine modern HfApi kullanilmaktadir.
Modern Kullanim: api = HfApi(); api.upload_folder(...)


## 8. Model Kartı (Model Card - README.md) Hazırlama
Bölüm 4.4'te açıklandığı gibi, paylaştığınız her modelin açıklayıcı bir `README.md` (model kartı) dosyasına sahip olması topluluk tarafından bulunabilmesi ve anlaşılabilmesi için şarttır.

In [10]:
model_card_content = """---
language: fr
tags:
- camembert
- fill-mask
license: mit
---

# CamemBERT Deneme Modeli (PyTorch)

Bu model, NLP Dersi Bölüm 4 uygulaması kapsamında Hugging Face Hub'a yüklenmiştir.

## Model Açıklaması
- **Geliştirici:** Emirhan Alemdaroğlu
- **Temel Model:** `camembert-base`
- **Görev:** Maskeli Dil Modelleme (Fill-Mask)
- **Dil:** Fransızca
- **Çatı:** PyTorch

## Kullanım Örneği
```python
from transformers import pipeline

mask_filler = pipeline("fill-mask", model="emirhanalemdaroglu/dummy-camembert-pt")
print(mask_filler("Le camembert est <mask /> :)"))
```
"""

with open("ornek_model_karti.md", "w", encoding="utf-8") as f:
    f.write(model_card_content)

print("Ornek Model Karti (ornek_model_karti.md) basariyla olusturuldu!")


Ornek Model Karti (ornek_model_karti.md) basariyla olusturuldu!
